In [2]:
import yfinance as yf
import pandas as pd
import fetch_stock_data as data
import stock_model_trainer as trainer
import stock_predictor_models as models
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from itertools import cycle

# Collect and Preprocess Data

In [3]:
# list of symbols
sp500_symbols = data.get_sp500_symbols()
# collect entire window (train, test, eval)
df_all = data.collect_data(sp500_symbols, "2015-01-01", "2025-10-31", freq="1d")
# drop any stocks missing any days, leaves 401 stocks
df_all = df_all.dropna(axis=1, how="any")
# convert to pct change
df_all = df_all.pct_change().dropna()
# normalize
mean_train = df_all.loc[df_all.index < "2024-01-01"].mean(axis=0)
std_train = df_all.loc[df_all.index < "2024-01-01"].std(axis=0)
df_all = (df_all - mean_train) / std_train
# train is 2015-01-01 to 2024-01-01
pandas_df_train = df_all.loc[df_all.index < "2024-01-01"]
# test is 2024-01-01 to 2024-10-31
pandas_df_test = df_all.loc[(df_all.index >= "2024-01-01") & (df_all.index <= "2024-10-31")]
# holdout is 2024-11-01 to 2025-10-31
pandas_df_holdout = df_all.loc[(df_all.index >= "2024-11-01") & (df_all.index <= "2025-10-31")]
# convert to torch tensors
prices_train = torch.tensor(pandas_df_train.values, dtype=torch.float32)
prices_test = torch.tensor(pandas_df_test.values, dtype=torch.float32)
prices_holdout = torch.tensor(pandas_df_holdout.values, dtype=torch.float32)

/Users/williamlusty/Documents/GitHub/CS7643-Final-Project/fetch_stock_data.py:31: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(symbols, start=start_date, end=end_date, interval=freq)
[*********************100%***********************]  505 of 505 completed

74 Failed downloads:
['APC', 'XEC', 'ABC', 'DISH', 'MON', 'CHK', 'HES', 'RHT', 'DISCA', 'KSU', 'CELG', 'FL', 'BRK.B', 'DWDP', 'RTN', 'ARNC', 'WLTW', 'AGN', 'VAR', 'PBCT', 'HRS', 'MRO', 'GPS', 'PKI', 'HCP', 'CERN', 'LLL', 'WRK', 'JWN', 'SYMC', 'TSS', 'BHGE', 'VIAB', 'RE', 'CTL', 'WBA', 'DFS', 'ETFC', 'MYL', 'JNPR', 'PXD', 'CTXS', 'JEC', 'ATVI', 'NBL', 'CXO', 'NLSN', 'PDCO', 'TIF', 'FLIR', 'ALXN', 'FBHS', 'CBS', 'COG', 'UTX', 'ANSS', 'ADS', 'DRE', 'BLL', 'XLNX', 'XL', 'TMK', 'DISCK', 'ANTM']: YFTzMissingError('possibly delisted; no timezone found')
['WYN', 'DPS', 'LUK', 'BF.B', 'GGP', 'SNI', 'KORS', 'HCN', 'SRCL', 'CBG']: YFPricesMissingError('possibly delisted; no price data found 

In [4]:
# need a map from the evaluation stocks to their column position in the data
eval_stocks = data.get_eval_stocks()
num_stocks_out = len(eval_stocks)
stock_index_map = {c: df_all.columns.get_loc(c) for c in eval_stocks}
stock_indices = [stock_index_map[stock] for stock in eval_stocks]

In [5]:
# hold onto mean and std of train for only the eval stocks
# this helps to undo transformation for just the eval stocks
eval_mean_train = mean_train.iloc[stock_indices].values
eval_std_train = std_train.iloc[stock_indices].values

In [6]:
# turn raw data into sequences
# X will be shape (batches, seq_len, num_stocks)
# y will be shape (batches, num_stocks)
seq_len = 60
X_train, y_train = trainer.create_sequences(prices_train, seq_len)
X_test, y_test = trainer.create_sequences(prices_test, seq_len)
X_holdout, y_holdout = trainer.create_sequences(prices_holdout, seq_len)
_, seq_len, num_stocks_in = X_train.shape

In [7]:
# select the eval columns
y_train = y_train[:, stock_indices]
y_test = y_test[:, stock_indices]
y_holdout = y_holdout[:, stock_indices]

# Train

In [137]:
# setup for training
criterion = nn.HuberLoss()
model = models.StockTransformer(
    input_dim=num_stocks_in,
    output_dim=num_stocks_out, # predict only the 5 we care about
    hidden_dim=16,
    dropout=0.5,
    num_layers=1,
)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
model_trainer = trainer.ModelTrainer(
    criterion,
    model,
    optimizer,
    X_train,
    y_train,
    X_test,
    y_test,
)

In [138]:
model_trainer.train(num_epochs=200)
model_trainer.plot_losses(fig_path="sp500_transformer/losses.png")

Epoch [5/200], Train Loss: 0.466732, Test Loss: 0.391473
Epoch [10/200], Train Loss: 0.453860, Test Loss: 0.380977
Epoch [15/200], Train Loss: 0.442976, Test Loss: 0.372524
Epoch [20/200], Train Loss: 0.438384, Test Loss: 0.365186
Epoch [25/200], Train Loss: 0.425036, Test Loss: 0.358802
Epoch [30/200], Train Loss: 0.419340, Test Loss: 0.352995
Epoch [35/200], Train Loss: 0.414824, Test Loss: 0.347569
Epoch [40/200], Train Loss: 0.406759, Test Loss: 0.342694
Epoch [45/200], Train Loss: 0.405934, Test Loss: 0.338516
Epoch [50/200], Train Loss: 0.399760, Test Loss: 0.334895
Epoch [55/200], Train Loss: 0.394520, Test Loss: 0.331718
Epoch [60/200], Train Loss: 0.392154, Test Loss: 0.328886
Epoch [65/200], Train Loss: 0.392176, Test Loss: 0.326371
Epoch [70/200], Train Loss: 0.385952, Test Loss: 0.324134
Epoch [75/200], Train Loss: 0.383865, Test Loss: 0.322220
Epoch [80/200], Train Loss: 0.380688, Test Loss: 0.320469
Epoch [85/200], Train Loss: 0.379100, Test Loss: 0.318831
Epoch [90/200],

<Figure size 640x480 with 0 Axes>

# Eval on Test Set

In [139]:
model.eval()
with torch.no_grad():
    pred_test_scaled = model(X_test)
pred_test_unscaled = pred_test_scaled * eval_std_train + eval_mean_train
y_test_unscaled = y_test * eval_std_train + eval_mean_train

In [140]:
# Get the default matplotlib color cycle
color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])
for i, stock in enumerate(eval_stocks):
    color = next(color_cycle)
    plt.plot(pred_test_unscaled[:, i], label=f"{stock}_Pred", color=color)
    plt.plot(y_test_unscaled[:, i], label=f"{stock}_Actual", color=color, linestyle="--")
    plt.title(f"{stock} Daily Returns Predicted vs Actual")
    plt.legend()
    plt.savefig(f"outputs/sp500_transformer/{stock}_test_preds.png")
    plt.clf()

<Figure size 640x480 with 0 Axes>

In [141]:
# MAE by stock
torch.mean(torch.abs(pred_test_unscaled - y_test_unscaled), dim=0)

tensor([0.0116, 0.0106, 0.0109, 0.0171, 0.0111], dtype=torch.float64)

In [142]:
eval_stocks

['AAPL', 'JPM', 'XOM', 'BA', 'UNH']

In [143]:
# MAE total
torch.mean(torch.abs(pred_test_unscaled - y_test_unscaled))

tensor(0.0123, dtype=torch.float64)

In [144]:
# directional accuracy by stock
(pred_test_unscaled * y_test_unscaled > 0).to(torch.float).mean(dim=0)

tensor([0.5099, 0.5497, 0.4768, 0.4371, 0.5430])

In [145]:
# directional accuracy overall
(pred_test_unscaled * y_test_unscaled > 0).to(torch.float).mean()

tensor(0.5033)

# Eval on Holdout set

In [146]:
model.eval()
with torch.no_grad():
    pred_holdout_scaled = model(X_holdout)
pred_holdout_unscaled = pred_holdout_scaled * eval_std_train + eval_mean_train
y_holdout_unscaled = y_holdout * eval_std_train + eval_mean_train

In [147]:
# Get the default matplotlib color cycle
color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])
for i, stock in enumerate(eval_stocks):
    color = next(color_cycle)
    plt.plot(pred_holdout_unscaled[:, i], label=f"{stock}_Pred", color=color)
    plt.plot(y_holdout_unscaled[:, i], label=f"{stock}_Actual", color=color, linestyle="--")
    plt.title(f"{stock} Daily Returns Predicted vs Actual")
    plt.legend()
    plt.savefig(f"outputs/sp500_transformer/{stock}_holdout_preds.png")
    plt.clf()

<Figure size 640x480 with 0 Axes>

In [148]:
# MAE by stock
torch.mean(torch.abs(pred_holdout_unscaled - y_holdout_unscaled), dim=0)

tensor([0.0148, 0.0117, 0.0123, 0.0169, 0.0195], dtype=torch.float64)

In [149]:
# MAE total
torch.mean(torch.abs(pred_test_unscaled - y_test_unscaled))

tensor(0.0123, dtype=torch.float64)

In [150]:
# directional accuracy by stock
(pred_holdout_unscaled * y_holdout_unscaled > 0).to(torch.float).mean(dim=0)

tensor([0.5079, 0.5291, 0.4868, 0.5132, 0.4815])

In [151]:
# directional accuracy overall
(pred_holdout_unscaled * y_holdout_unscaled > 0).to(torch.float).mean()

tensor(0.5037)